# Logistic Regression with Missing Labels — Example Usage

This notebook demonstrates the full pipeline:
1. Data loading and preprocessing
2. FISTA L1-regularized logistic regression
3. Missing label simulation (MCAR, MAR1, MAR2, MNAR)
4. Semi-supervised methods: Oracle, Naive, KNN (hard/proba), EM
5. Full experiment across missing rates and schemas

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score

from utils.logistic_lasso_regression_fista import LogisticLassoRegressionFISTA
from utils.missing_schemas import MCAR, MAR1, MAR2, MNAR, generate_missing_y
from utils.unlabeled_log_reg import UnlabeledLogReg, OracleLogReg, NaiveLogReg

## 1. Load and Prepare Data

We use the breast cancer dataset from sklearn as a convenient binary classification example.
Replace this with your own dataset loaded via `fetch_data.ipynb`.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Class balance: {y.value_counts().to_dict()}")
X.head()

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# further split train into train + validation
X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(X_train_raw, y_train, test_size=0.2, stratify=y_train, random_state=42)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)
X_valid = scaler.transform(X_valid_raw)
X_test  = scaler.transform(X_test_raw)

print(f"Train size: {X_train.shape[0]}, Valid size: {X_valid.shape[0]}, Test size: {X_test.shape[0]}")

## 2. FISTA L1-Regularized Logistic Regression

### 2.1 Initialize and fit

In [ ]:
model = LogisticLassoRegressionFISTA(
    lambdas=None, # default log grid: 1e-4 to 10
    measure="roc_auc", # metric for lambda selection
    max_iter=1000,
    stop_condition=1e-6,
)

model.fit(X_train, y_train.values)
print("Fitting done — coefficient paths computed for all lambdas.")

### 2.2 Select best lambda on validation set

In [ ]:
best_score = model.validate(X_valid, y_valid.values)

print(f"Best validation ROC AUC: {best_score:.4f}")
print(f"Selected lambda: {model.best_lambda_:.6f}")

### 2.3 Evaluate on test set

In [ ]:
y_proba = model.predict_proba(X_test)
y_pred  = (y_proba >= 0.5).astype(int)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC AUC : {roc_auc_score(y_test, y_proba):.4f}")
print(f"Non-zero coefficients: {np.sum(np.abs(model.beta_) > 1e-8)} / {len(model.beta_)}")

### 2.4 Visualizations

In [ ]:
# ROC AUC vs lambda
model.plot(X_valid, y_valid.values)

In [ ]:
# coefficient paths
model.plot_coefficients()

## 3. Missing Label Simulation

Available schemas:
- **MCAR** — labels are masked uniformly at random
- **MAR1** — missingness depends on a single feature (first column)
- **MAR2** — missingness depends on all features
- **MNAR** — missingness depends on both features and the true label

In [ ]:
X_train_df = pd.DataFrame(X_train)
y_train_s  = pd.Series(y_train.values)

missing_rate = 0.5

for schema_name, fn in [("MCAR", MCAR), ("MAR1", MAR1), ("MAR2", MAR2), ("MNAR", MNAR)]:
    _, y_miss = fn(X_train_df, y_train_s, missing_rate)
    n_missing = (y_miss == -1).sum()
    print(f"{schema_name}: {n_missing} missing labels out of {len(y_miss)} ({n_missing/len(y_miss):.1%})")

In [ ]:
#alternatively use generate_missing_y(X_train_df, y_train_s, schema="MAR1", missing_rate=0.5)
_, y_mcar = generate_missing_y(X_train_df, y_train_s, scheme="MCAR", missing_rate=0.5)
print("y_mcar value counts:", y_mcar.value_counts().to_dict())

## 4. Semi-supervised Methods

We now apply MCAR with 50% missing rate and compare all five methods.

In [ ]:
_, y_train_miss = generate_missing_y(X_train_df, y_train_s, scheme="MCAR", missing_rate=0.5)

print(f"Labeled: {(y_train_miss != -1).sum()}")
print(f"Unlabeled: {(y_train_miss == -1).sum()}")

### 4.1 Oracle — trained on all labels (upper bound)

In [ ]:
oracle = OracleLogReg()
oracle.fit(X_train, y_train.values, X_valid, y_valid.values)

res_oracle = oracle.evaluate(X_test, y_test.values, dataset_name="BreastCancer", missing_schema="MCAR")
res_oracle

### 4.2 Naive — trained only on labeled samples (lower bound)

In [ ]:
naive = NaiveLogReg()
naive.fit(X_train, y_train_miss.values, X_valid, y_valid.values)

res_naive = naive.evaluate(X_test, y_test.values, dataset_name="BreastCancer", missing_schema="MCAR")
res_naive

### 4.3 KNN — hard label imputation

In [ ]:
knn_hard = UnlabeledLogReg(method="KNN", label_type="hard", n_neighbors=5)
knn_hard.fit(X_train, y_train_miss.values, X_valid, y_valid.values)

res_knn_hard = knn_hard.evaluate(X_test, y_test.values, dataset_name="BreastCancer", missing_schema="MCAR")
res_knn_hard

### 4.4 KNN — probabilistic (soft) label imputation

In [ ]:
knn_proba = UnlabeledLogReg(method="KNN", label_type="proba", n_neighbors=5)
knn_proba.fit(X_train, y_train_miss.values, X_valid, y_valid.values)

res_knn_proba = knn_proba.evaluate(X_test, y_test.values, dataset_name="BreastCancer", missing_schema="MCAR")
res_knn_proba

### 4.5 EM — Expectation-Maximization

In [ ]:
em = UnlabeledLogReg(method="EM", max_em_iter=10, n_neighbors=5)
em.fit(X_train, y_train_miss.values, X_valid, y_valid.values)

res_em = em.evaluate(X_test, y_test.values, dataset_name="BreastCancer", missing_schema="MCAR")
res_em

### 4.6 Results comparison

In [ ]:
results = pd.DataFrame([res_oracle, res_naive, res_knn_hard, res_knn_proba, res_em])
results = results[["method", "accuracy", "balanced_accuracy", "f1_score", "roc_auc"]]
results = results.set_index("method").round(4)
results

## 5. Full Experiment Across Missing Rates and Schemas

In [ ]:
from utils.experiments import run_full_experiment

# use the full (unscaled) X and y — scaling happens inside run_single_fold
results_df = run_full_experiment(X=X, y=y, dataset_name="BreastCancer", missing_rates=[0.3, 0.5, 0.8])
results_df.head(10)

In [ ]:
# aggregate: mean ROC AUC per method and missing rate
summary = (
    results_df
    .groupby(["method", "missing_rate"])["roc_auc"]
    .mean()
    .unstack("missing_rate")
    .round(4)
)
summary

In [ ]:
# aggregate: mean ROC AUC per method and schema
summary_schema = (
    results_df
    .groupby(["method", "missing_schema"])["roc_auc"]
    .mean()
    .unstack("missing_schema")
    .round(4)
)
summary_schema

## 6. Parallel Experiment (optional)

For larger datasets, use the parallelised version. Requires `joblib` and `threadpoolctl`.

In [ ]:
from utils.experiments import run_full_experiment_parallel

results_parallel = run_full_experiment_parallel(X=X, y=y, dataset_name="BreastCancer", missing_rates=[0.3, 0.5, 0.8])
print(f"Total results: {len(results_parallel)}")
results_parallel.head()